# Chatty 🤖 — a tiny, silly chatbot in Keras

A word-level **sequence-to-sequence LSTM with attention**, trained from scratch
in a few minutes on a laptop.

**Memory.** The model doesn't just see your last message. Its input is the
last **3 exchanges + your new message**. If you tell it your name and ask two
turns later, the answer is somewhere in its input, and attention has to find
it. Anything older than that window is gone, and Chatty admits it.

```
<u> my name is olga <b> nice to meet you , olga ! <u> tell me a joke <b> ... <u> what's my name ?
└────────────────────────────── encoder input (context) ─────────────────────────────────────┘
                                                                  decoder output → your name is olga !
```

**Plan**
1. Generate silly synthetic conversations (there's no dataset for *this* bot's personality).
2. Turn them into (context → reply) pairs.
3. Build and train an encoder–decoder LSTM with dot-product attention.
4. Check it: learning curves, a memory test, an attention heatmap.
5. Chat with it.

In [1]:
import json
import os
import random
import re
from collections import Counter
from pathlib import Path

os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import warnings

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from keras import layers, ops

SEED = 7
keras.utils.set_random_seed(SEED)
MODEL_DIR = Path("model")
# the encoder output carries a Keras mask; the step decoder gets the mask explicitly, so this warning is noise
warnings.filterwarnings("ignore", message=".*does not support masking.*")
print("keras", keras.__version__, "| backend:", keras.backend.backend())

keras 3.15.1 | backend: tensorflow


## 1. Synthetic conversations

Three kinds of exchanges get mixed into conversations of 3–10 turns:

- **Chit-chat:** greetings, jokes, "are you a robot?", "i'm bored". Replies are
  deterministic per phrasing, so the model has one right answer to learn.
- **Telling facts:** name, favourite food, pet, city.
- **Asking for facts later.** The right reply is computed from what's
  **inside the memory window** (the last 3 exchanges):
  - fact told within the window → *"your name is olga !"* (latest value wins, so corrections work)
  - not in the window → *"hmm ... i don't remember . tell me again ?"*

Jokes are context-dependent too: "another joke" continues from the last joke in
the window.

In [2]:
MEMORY = 3  # previous exchanges the bot can see

NAMES = ["olga", "ivan", "anna", "vlad", "maria", "alex", "sam", "lena", "max", "nina", "omar", "sofia", "leo",
         "mia", "dmitri", "kate", "john", "emma", "lucas", "yuki", "aisha", "pablo", "zoe", "ben", "elena",
         "igor", "lily", "noah", "sara", "tom", "irina", "jack", "chloe", "ali", "eva", "kai", "nora", "oscar",
         "masha", "pavel", "rita", "felix", "hana", "luis", "ella", "artem", "mila", "david", "julia", "petr",
         "katerina", "catherina"]
FOODS = ["pizza", "sushi", "pasta", "tacos", "borscht", "pancakes", "ramen", "burgers", "dumplings", "salad",
         "curry", "waffles", "soup", "cheese", "chocolate", "ice cream", "noodles", "pierogi", "kebab", "falafel"]
PETS = ["cat", "dog", "hamster", "parrot", "turtle", "rabbit", "goldfish", "lizard", "snake", "ferret",
        "hedgehog", "pony"]
CITIES = ["london", "paris", "berlin", "tokyo", "moscow", "kyiv", "warsaw", "madrid", "rome", "prague",
          "vienna", "oslo", "lisbon", "dubai", "toronto", "sydney", "boston", "chicago", "minsk", "riga"]

# slot -> values, how the user tells it (with the bot's ack), how they ask (with the bot's recall)
SLOTS = {
    "name": dict(values=NAMES,
                 tell=[("my name is {v}", "nice to meet you , {v} !"),
                       ("i am {v}", "hello {v} ! what a cool name"),
                       ("call me {v}", "okay {v} , i will try to remember"),
                       ("actually my name is {v}", "oops , sorry {v} !")],
                 ask=[("what is my name ?", "your name is {v} !"),
                      ("what's my name ?", "you are {v} , of course !"),
                      ("do you remember my name ?", "yes ! {v} !"),
                      ("who am i ?", "you are {v} , my favorite human")]),
    "food": dict(values=FOODS,
                 tell=[("i love {v}", "{v} ? yummy !"),
                       ("my favorite food is {v}", "mmm , {v} is the best"),
                       ("i could eat {v} every day", "{v} every day ? respect")],
                 ask=[("what food do i like ?", "you love {v} !"),
                      ("what is my favorite food ?", "{v} ! you told me"),
                      ("what do i like to eat ?", "{v} , obviously")]),
    "pet": dict(values=PETS,
                tell=[("i have a {v}", "a {v} ! so cute"),
                      ("my pet is a {v}", "wow , a {v} ! give it a hug from me")],
                ask=[("what pet do i have ?", "you have a {v} !"),
                     ("do you remember my pet ?", "of course , your {v} !")]),
    "city": dict(values=CITIES,
                 tell=[("i live in {v}", "{v} ! i heard it is lovely"),
                       ("i am from {v}", "ooh , {v} . send me a postcard")],
                 ask=[("where do i live ?", "you live in {v} !"),
                      ("where am i from ?", "you are from {v} , right ?")]),
}
DONT_KNOW = "hmm ... i don't remember . tell me again ?"

JOKES = ["why did the robot go on holiday ? to recharge its batteries !",
         "what do you call a sleeping bull ? a bulldozer !",
         "why was the math book sad ? it had too many problems",
         "i told my computer a joke . it did not laugh , it just crashed"]
JOKE_ASKS = ["tell me a joke", "say something funny", "make me laugh", "another joke", "one more joke"]

# intent: [(user phrasing, bot reply), ...]
CHITCHAT = {
    "greet": [("hi", "hi there , human !"), ("hello", "hello ! ready for some nonsense ?"),
              ("hey", "hey hey hey !"), ("good morning", "good morning , sunshine !")],
    "how": [("how are you ?", "i'm great , i just had a byte to eat"),
            ("how are you doing ?", "feeling bouncy ! and you ?"),
            ("what's up ?", "the sky . and my mood !")],
    "fine": [("i'm good", "glad to hear it !"), ("i'm fine", "fine is fine !"), ("great", "great is great !")],
    "who": [("what's your name ?", "i'm chatty , a very small and very silly bot"),
            ("who are you ?", "i'm chatty ! i was trained five minutes ago")],
    "robot": [("are you a robot ?", "beep boop . i mean , no . i mean , yes ."),
              ("are you human ?", "i'm a bunch of numbers pretending to be fun")],
    "sad": [("i'm sad", "oh no ! here is a virtual cookie"), ("bad day", "sending you a big digital hug")],
    "happy": [("i'm happy", "yay ! happy dance !"), ("good day", "that makes my circuits warm")],
    "bored": [("i'm bored", "let's count to infinity . one , two ..."), ("this is boring", "rude ! but fair")],
    "thanks": [("thanks", "you're welcome !"), ("thank you", "anytime , friend")],
    "bye": [("bye", "bye bye ! don't forget me"), ("see you", "see you , space cowboy")],
    "weather": [("how is the weather ?", "i live in a computer , it is always warm here")],
    "friends": [("do you like me ?", "of course ! you're my favorite human today"),
                ("are we friends ?", "best friends forever !")],
    "food_bot": [("what do you eat ?", "i eat electricity , with a little salt")],
    "age": [("how old are you ?", "i was born a few minutes ago , during training")],
    "help": [("what can you do ?", "i chat , i joke , and i remember things . for a little while")],
    "filler": [("ok", "cool cool cool"), ("lol", "i know , i'm hilarious"), ("haha", "haha , right ?"),
               ("nice", "nice nice")],
}


# the same bot in Russian: nominative-only templates, so the values never need declining; no "ё"
NAMES_RU = ["ольга", "иван", "анна", "влад", "мария", "саша", "катя", "катерина", "лена", "макс", "нина", "софия",
            "лев", "миша", "дима", "кира", "егор", "маша", "павел", "рита", "артем", "мила", "денис", "юля",
            "петр", "игорь", "лиля", "ника", "вера", "олег"]
FOODS_RU = ["пицца", "суши", "паста", "борщ", "блины", "рамен", "бургеры", "пельмени", "салат", "карри",
            "вафли", "суп", "сыр", "шоколад", "мороженое", "лапша", "вареники", "шашлык", "плов", "сырники"]
PETS_RU = ["кот", "кошка", "собака", "хомяк", "попугай", "черепаха", "кролик", "рыбка", "ящерица", "змея",
           "хорек", "ежик"]
CITIES_RU = ["москва", "питер", "киев", "минск", "казань", "сочи", "рига", "варшава", "берлин", "париж",
             "лондон", "токио", "рим", "прага", "вена", "осло", "дубай", "новосибирск", "екатеринбург", "тбилиси"]

SLOTS_RU = {
    "name": dict(values=NAMES_RU,
                 tell=[("меня зовут {v}", "приятно познакомиться , {v} !"),
                       ("я {v}", "привет , {v} ! классное имя"),
                       ("зови меня {v}", "хорошо , {v} , постараюсь запомнить"),
                       ("вообще-то меня зовут {v}", "ой , прости , {v} !")],
                 ask=[("как меня зовут ?", "тебя зовут {v} !"),
                      ("помнишь мое имя ?", "да ! {v} !"),
                      ("ты помнишь , как меня зовут ?", "конечно , {v} !"),
                      ("кто я ?", "ты {v} , мой любимый человек")]),
    "food": dict(values=FOODS_RU,
                 tell=[("моя любимая еда {v}", "{v} ? вкуснятина !"),
                       ("лучшая еда это {v}", "м-м-м , {v} это топ"),
                       ("обожаю , когда на ужин {v}", "{v} на ужин ? уважаю")],
                 ask=[("какая у меня любимая еда ?", "{v} ! ты сам говорил"),
                      ("что я люблю есть ?", "{v} , конечно"),
                      ("какую еду я люблю ?", "твоя любимая еда {v} !")]),
    "pet": dict(values=PETS_RU,
                tell=[("у меня есть {v}", "{v} ! какая прелесть"),
                      ("мой питомец {v}", "ого , {v} ! обними от меня")],
                ask=[("кто у меня есть ?", "у тебя есть {v} !"),
                     ("помнишь моего питомца ?", "конечно , твой питомец {v} !")]),
    "city": dict(values=CITIES_RU,
                 tell=[("мой город {v}", "{v} ! говорят , там классно"),
                       ("я из города {v}", "о , {v} . пришли мне открытку")],
                 ask=[("где я живу ?", "твой город {v} !"),
                      ("откуда я ?", "ты из города {v} , да ?")]),
}
DONT_KNOW_RU = "хм ... не помню . напомни ?"

JOKES_RU = ["почему робот ушел в отпуск ? чтобы подзарядить батарейки !",
            "что сказал ноль восьмерке ? классный ремень !",
            "почему компьютер пошел к врачу ? у него был вирус",
            "я рассказал компьютеру шутку . он не засмеялся , а просто завис"]
JOKE_ASKS_RU = ["расскажи шутку", "скажи что-нибудь смешное", "рассмеши меня", "еще шутку", "еще одну шутку"]

CHITCHAT_RU = {
    "greet": [("привет", "привет , человек !"), ("здравствуй", "здравствуй ! готов к глупостям ?"),
              ("хай", "хай-хай-хай !"), ("доброе утро", "доброе утро , солнышко !")],
    "how": [("как дела ?", "отлично , только что перекусил байтом"), ("как ты ?", "бодрячком ! а ты ?"),
            ("что нового ?", "новая версия меня . шучу !")],
    "fine": [("нормально", "нормально это нормально !"), ("хорошо", "рад слышать !"), ("отлично", "отлично это отлично !")],
    "who": [("как тебя зовут ?", "я болтун , очень маленький и очень глупый бот"),
            ("кто ты ?", "я болтун ! меня обучили пять минут назад")],
    "robot": [("ты робот ?", "бип-буп . то есть нет . то есть да ."),
              ("ты человек ?", "я кучка чисел , которая притворяется веселой")],
    "sad": [("мне грустно", "о нет ! держи виртуальную печеньку"), ("плохой день", "отправляю большие цифровые обнимашки")],
    "happy": [("я счастлив", "ура ! танец счастья !"), ("хороший день", "от этого греются мои микросхемы")],
    "bored": [("мне скучно", "давай считать до бесконечности . раз , два ..."), ("скучно", "грубо ! но честно")],
    "thanks": [("спасибо", "пожалуйста !"), ("благодарю", "всегда рад , друг")],
    "bye": [("пока", "пока-пока ! не забывай меня"), ("до встречи", "до встречи , космический ковбой")],
    "weather": [("какая погода ?", "я живу в компьютере , тут всегда тепло")],
    "friends": [("я тебе нравлюсь ?", "конечно ! ты сегодня мой любимый человек"),
                ("мы друзья ?", "лучшие друзья навсегда !")],
    "food_bot": [("что ты ешь ?", "я ем электричество , с щепоткой соли")],
    "age": [("сколько тебе лет ?", "я родился пару минут назад , во время обучения")],
    "help": [("что ты умеешь ?", "болтаю , шучу и запоминаю . ненадолго")],
    "filler": [("ок", "клево-клево-клево"), ("лол", "знаю , я уморительный"), ("хаха", "хаха , да ?"),
               ("круто", "круто-круто")],
}

# each conversation is in one language, picked at random
LANGS = {
    "en": dict(slots=SLOTS, chitchat=CHITCHAT, jokes=JOKES, joke_asks=JOKE_ASKS, dont_know=DONT_KNOW),
    "ru": dict(slots=SLOTS_RU, chitchat=CHITCHAT_RU, jokes=JOKES_RU, joke_asks=JOKE_ASKS_RU, dont_know=DONT_KNOW_RU),
}


def recall(slot: str, window: list) -> str | None:
    """Latest value of `slot` told within the memory window, or None."""
    for ex in reversed(window):
        if ex.get("slot") == slot and ex["kind"] == "tell":
            return ex["value"]
    return None


def next_joke(window: list, n_jokes: int) -> int:
    for ex in reversed(window):
        if ex["kind"] == "joke":
            return (ex["joke"] + 1) % n_jokes
    return 0


def make_conversation(rng: random.Random) -> list[dict]:
    L = LANGS[rng.choice(list(LANGS))]
    convo = []
    for turn in range(rng.randint(3, 10)):
        window = convo[-MEMORY:]
        roll = rng.random()
        if turn == 0 and roll < 0.5:
            user, bot = rng.choice(L["chitchat"]["greet"])
            ex = {"kind": "chat"}
        elif roll < 0.30:                                   # tell a fact
            slot = rng.choice(list(L["slots"]))
            value = rng.choice(L["slots"][slot]["values"])
            user_t, bot_t = rng.choice(L["slots"][slot]["tell"])
            user, bot = user_t.format(v=value), bot_t.format(v=value)
            ex = {"kind": "tell", "slot": slot, "value": value}
        elif roll < 0.55:                                   # ask for a fact
            told = [e["slot"] for e in window if e["kind"] == "tell"]
            slot = rng.choice(told) if told and rng.random() < 0.7 else rng.choice(list(L["slots"]))
            user_t, bot_t = rng.choice(L["slots"][slot]["ask"])
            value = recall(slot, window)
            user, bot = user_t, (bot_t.format(v=value) if value else L["dont_know"])
            ex = {"kind": "ask", "slot": slot, "remembered": value is not None}
        elif roll < 0.65:                                   # joke
            j = next_joke(window, len(L["jokes"]))
            user, bot = rng.choice(L["joke_asks"]), L["jokes"][j]
            ex = {"kind": "joke", "joke": j}
        else:                                               # chit-chat
            user, bot = rng.choice(L["chitchat"][rng.choice(list(L["chitchat"]))])
            ex = {"kind": "chat"}
        convo.append({**ex, "user": user, "bot": bot})
    return convo


rng = random.Random(SEED)
for ex in make_conversation(rng):
    print(f"🧑 {ex['user']:<32} 🤖 {ex['bot']}")

🧑 привет                           🤖 привет , человек !
🧑 меня зовут софия                 🤖 приятно познакомиться , софия !
🧑 я счастлив                       🤖 ура ! танец счастья !
🧑 мой город киев                   🤖 киев ! говорят , там классно
🧑 мой город питер                  🤖 питер ! говорят , там классно


## 2. From conversations to training pairs

Every bot turn becomes one example:

- **input (context):** `<u> ... <b> ...` for the last 3 exchanges, then `<u>` + the new message
- **target:** the bot's reply, wrapped in `<start> ... <end>`

The decoder learns with *teacher forcing*: it's fed the correct reply shifted
by one token and must predict the next token at each step.

In [3]:
TOKEN_RE = re.compile(r"[a-zа-я0-9]+(?:'[a-z]+)?|[^\sa-zа-я0-9]")
PAD, UNK, START, END, USER, BOT = "<pad>", "<unk>", "<start>", "<end>", "<u>", "<b>"


def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(text.lower().replace("ё", "е"))  # the data has no "ё"


def context_tokens(history: list[dict], message: str) -> list[str]:
    toks = []
    for ex in history[-MEMORY:]:
        toks += [USER, *tokenize(ex["user"]), BOT, *tokenize(ex["bot"])]
    return toks + [USER, *tokenize(message)]


def to_pairs(convos):
    pairs = []
    for convo in convos:
        for i, ex in enumerate(convo):
            pairs.append({"context": context_tokens(convo[:i], ex["user"]), "reply": tokenize(ex["bot"]),
                          "kind": ex["kind"], "remembered": ex.get("remembered")})
    return pairs


rng = random.Random(SEED)
train_pairs = to_pairs(make_conversation(rng) for _ in range(10000))
val_pairs = to_pairs(make_conversation(rng) for _ in range(1000))

counts = Counter(t for p in train_pairs for t in p["context"] + p["reply"])
vocab = [PAD, UNK, START, END] + sorted(counts)          # <u>/<b> are already in counts
word2id = {w: i for i, w in enumerate(vocab)}
MAX_CONTEXT = max(len(p["context"]) for p in train_pairs)
MAX_REPLY = max(len(p["reply"]) for p in train_pairs) + 1   # +1 for <start>/<end>

print(f"{len(train_pairs):,} train pairs, {len(val_pairs):,} val pairs")
print(f"vocab: {len(vocab)} words | longest context: {MAX_CONTEXT} tokens | longest reply: {MAX_REPLY - 1}")
pd.Series(Counter(p["kind"] for p in train_pairs)).plot.bar(title="training pairs by kind", rot=0, figsize=(6, 2.5))
plt.show()

64,892 train pairs, 6,440 val pairs
vocab: 623 words | longest context: 71 tokens | longest reply: 15


/var/folders/7p/wxlym5j148z5bvgtpk6tb9tm0000gn/T/ipykernel_88422/1130620568.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
def encode(tokens, length):
    ids = [word2id.get(t, word2id[UNK]) for t in tokens][-length:]   # keep the most recent tokens
    return ids + [0] * (length - len(ids))


def vectorize(pairs):
    ctx = np.array([encode(p["context"], MAX_CONTEXT) for p in pairs], dtype="int32")
    dec_in = np.array([encode([START] + p["reply"], MAX_REPLY) for p in pairs], dtype="int32")
    dec_out = np.array([encode(p["reply"] + [END], MAX_REPLY) for p in pairs], dtype="int32")
    return (ctx, dec_in), dec_out


X_train, y_train = vectorize(train_pairs)
X_val, y_val = vectorize(val_pairs)

p = train_pairs[3]
print("context :", " ".join(p["context"]))
print("dec in  :", " ".join(vocab[i] for i in X_train[1][3] if i))
print("dec out :", " ".join(vocab[i] for i in y_train[3] if i))

context : <u> привет <b> привет , человек ! <u> меня зовут софия <b> приятно познакомиться , софия ! <u> я счастлив <b> ура ! танец счастья ! <u> мой город киев
dec in  : <start> киев ! говорят , там классно
dec out : киев ! говорят , там классно <end>


## 3. The model

```
context ─► Embedding ─► LSTM encoder ──► all encoder states ──┐
                              │ (final h, c)                  │
                              ▼                               ▼
reply_in ─► Embedding ─► LSTM decoder ─► decoder states ─► Attention ─► concat ─► Dense(tanh) ─► Dense(softmax over vocab)
```

- The **encoder LSTM** reads the whole context. Its final state starts the decoder.
- The **decoder LSTM** writes the reply one token at a time.
- **Attention** lets each decoder step look back at *every* context token. This
  is how "olga" from three turns ago ends up in the answer.
- Padding is masked everywhere, and the loss ignores padded positions.

In [5]:
EMBED_DIM, UNITS = 96, 192
V = len(vocab)

# shared layers (reused later by the step-by-step inference models)
embed = layers.Embedding(V, EMBED_DIM, name="embedding")
encoder_lstm = layers.LSTM(UNITS, return_sequences=True, return_state=True, name="encoder_lstm")
decoder_lstm = layers.LSTM(UNITS, return_sequences=True, return_state=True, name="decoder_lstm")
attention = layers.Attention(use_scale=True, name="attention")
combine = layers.Dense(UNITS, activation="tanh", name="combine")
to_vocab = layers.Dense(V, activation="softmax", name="next_word")


def decode_step(dec_tokens, enc_seq, enc_mask, state):
    dec_mask = ops.not_equal(dec_tokens, 0)
    dec_seq, h, c = decoder_lstm(embed(dec_tokens), initial_state=state, mask=dec_mask)
    ctx, scores = attention([dec_seq, enc_seq], mask=[dec_mask, enc_mask], return_attention_scores=True)
    probs = to_vocab(combine(layers.concatenate([dec_seq, ctx])))
    return probs, h, c, scores


context_in = keras.Input((MAX_CONTEXT,), dtype="int32", name="context")
reply_in = keras.Input((MAX_REPLY,), dtype="int32", name="reply_in")
context_mask = ops.not_equal(context_in, 0)
enc_seq, enc_h, enc_c = encoder_lstm(embed(context_in), mask=context_mask)
probs, _, _, _ = decode_step(reply_in, enc_seq, context_mask, [enc_h, enc_c])

model = keras.Model([context_in, reply_in], probs, name="chatty")
model.compile(optimizer=keras.optimizers.Adam(2e-3),
              loss=keras.losses.SparseCategoricalCrossentropy(ignore_class=0))
model.summary()

Model: "chatty"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ reply_in            │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context             │ (None, 71)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 16, 96)    │     59,808 │ context[0][0],    │
│ (Embedding)         │                   │            │ reply_in[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 71)        │          0 │ context[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 71, 192), │    221,952 │ embedding[0][0],  │
│                     │ (None, 192),      │            │ not_equal[0][0]   │
│                     │ (None, 192)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 16)        │          0 │ reply_in[0][0]    │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 16, 192), │    221,952 │ embedding[1][0],  │
│                     │ (None, 192),      │            │ encoder_lstm[0][… │
│                     │ (None, 192)]      │            │ encoder_lstm[0][… │
│                     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ [(None, 16, 192), │          1 │ decoder_lstm[0][… │
│ (Attention)         │ (None, 16, 71)]   │            │ encoder_lstm[0][… │
│                     │                   │            │ not_equal_1[0][0… │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16, 384)   │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combine (Dense)     │ (None, 16, 192)   │     73,920 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ next_word (Dense)   │ (None, 16, 623)   │    120,239 │ combine[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 697,872 (2.66 MB)

 Trainable params: 697,872 (2.66 MB)

 Non-trainable params: 0 (0.00 B)

## 4. Train

Early stopping watches the validation loss and keeps the best weights.

In [6]:
EPOCHS = int(os.environ.get("CHATTY_EPOCHS", 40))

history = model.fit(
    X_train, y_train, validation_data=(X_val, y_val), batch_size=64, epochs=EPOCHS,
    callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
               keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2)],
    verbose=2,
)

pd.DataFrame(history.history)[["loss", "val_loss"]].plot(title="cross-entropy per token", logy=True, figsize=(7, 3))
plt.xlabel("epoch")
plt.show()

Epoch 1/40


1014/1014 - 81s - 80ms/step - loss: 0.8769 - val_loss: 0.1448 - learning_rate: 0.0020


Epoch 2/40


1014/1014 - 89s - 88ms/step - loss: 0.0833 - val_loss: 0.0368 - learning_rate: 0.0020


Epoch 3/40


1014/1014 - 100s - 99ms/step - loss: 0.0243 - val_loss: 0.0166 - learning_rate: 0.0020


... (67 outputs left out)

1014/1014 - 111s - 110ms/step - loss: 6.3727e-05 - val_loss: 7.5688e-04 - learning_rate: 1.9531e-06


Epoch 38/40


1014/1014 - 116s - 115ms/step - loss: 6.3724e-05 - val_loss: 7.5691e-04 - learning_rate: 9.7656e-07


/var/folders/7p/wxlym5j148z5bvgtpk6tb9tm0000gn/T/ipykernel_88422/1773477501.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Talking to it: inference

Training feeds the decoder the correct previous word. In a real chat there is
no correct answer to feed it, so the decoder runs one step at a time on its
**own** previous word. That needs two small models built from the *same
trained layers*: an encoder model, and a single-step decoder model.

In [7]:
encoder_model = keras.Model(context_in, [enc_seq, enc_h, enc_c], name="encoder")

tok_in = keras.Input((1,), dtype="int32", name="token")
seq_in = keras.Input((MAX_CONTEXT, UNITS), name="encoder_states")
mask_in = keras.Input((MAX_CONTEXT,), dtype="bool", name="context_mask")
h_in, c_in = keras.Input((UNITS,), name="h"), keras.Input((UNITS,), name="c")
step_probs, step_h, step_c, step_scores = decode_step(tok_in, seq_in, mask_in, [h_in, c_in])
decoder_model = keras.Model([tok_in, seq_in, mask_in, h_in, c_in], [step_probs, step_h, step_c, step_scores],
                            name="decoder_step")


def detokenize(tokens):
    text = " ".join(tokens)
    return re.sub(r" ([,.!?])", r"\1", text).replace(" - ", "-")  # "бип - буп" -> "бип-буп"


def generate_batch(contexts, temperature=0.0, rng=np.random.default_rng()):
    # list of context token lists -> list of (reply tokens, attention [reply_len, context_len])
    ctx = np.array([encode(c, MAX_CONTEXT) for c in contexts], dtype="int32")
    seq, h, c = encoder_model(ops.convert_to_tensor(ctx), training=False)
    mask = ops.convert_to_tensor(ctx != 0)
    n = len(contexts)
    token = np.full((n, 1), word2id[START], dtype="int32")
    outs, attns, done = [[] for _ in range(n)], [[] for _ in range(n)], np.zeros(n, dtype=bool)
    for _ in range(MAX_REPLY + 5):
        p, h, c, s = decoder_model([ops.convert_to_tensor(token), seq, mask, h, c], training=False)
        p, s = ops.convert_to_numpy(p)[:, -1], ops.convert_to_numpy(s)[:, -1]
        if temperature > 0:
            p = np.exp(np.log(p + 1e-9) / temperature)
            nxt = np.array([rng.choice(len(row), p=row / row.sum()) for row in p])
        else:
            nxt = p.argmax(-1)
        for i in range(n):
            if done[i]:
                continue
            if nxt[i] == word2id[END]:
                done[i] = True
            else:
                outs[i].append(vocab[nxt[i]])
                attns[i].append(s[i])
        if done.all():
            break
        token = nxt.reshape(n, 1).astype("int32")
    return [(o, np.array(a)) for o, a in zip(outs, attns)]


def generate(context, temperature=0.0):
    return generate_batch([context], temperature)[0]


print(detokenize(generate(context_tokens([], "hello"))[0]))

hello! ready for some nonsense?


## 6. Does it actually work? Memory test

Fresh conversations the model has never seen. For every bot turn we check if
the generated reply matches the expected one exactly, split by kind:

- **chat / joke / tell:** plain chit-chat, jokes in order, acknowledging a fact
- **ask → remembered:** the fact is inside the window, so it must be recalled correctly
- **ask → forgotten:** the fact isn't in the window, so it must say it doesn't remember

In [8]:
rng_test = random.Random(12345)
test_convos = [make_conversation(rng_test) for _ in range(150)]

cases = [(context_tokens(convo[:i], ex["user"]), ex) for convo in test_convos for i, ex in enumerate(convo)]
replies = [r for start in range(0, len(cases), 256)
           for r, _ in generate_batch([c for c, _ in cases[start:start + 256]])]

rows = []
for (_, ex), reply in zip(cases, replies):
    kind = ex["kind"] if ex["kind"] != "ask" else ("ask → remembered" if ex["remembered"] else "ask → forgotten")
    rows.append({"kind": kind, "correct": reply == tokenize(ex["bot"]),
                 "user said": ex["user"], "expected": ex["bot"], "got": detokenize(reply)})
results = pd.DataFrame(rows)

summary = results.groupby("kind")["correct"].agg(["mean", "count"]).rename(columns={"mean": "exact match"})
summary.loc["overall"] = [results.correct.mean(), len(results)]
display(summary.round(3))
results[~results.correct].head(10)

,exact match,count
kind,,
ask → forgotten,1.0,133
ask → remembered,1.0,68
chat,1.0,448
joke,1.0,109
tell,1.0,241
overall,1.0,999


,kind,correct,user said,expected,got


### Where does it look? Attention heatmap

Rows are the words of the reply; columns are the context tokens. When Chatty
writes the name, attention should light up on the name from earlier turns.

In [9]:
demo_history = [
    {"user": "my name is olga", "bot": "nice to meet you , olga !"},
    {"user": "i have a parrot", "bot": "a parrot ! so cute"},
    {"user": "tell me a joke", "bot": JOKES[0]},
]
ctx = context_tokens(demo_history, "what's my name ?")
reply, attn = generate(ctx)
print("reply:", detokenize(reply))

fig, ax = plt.subplots(figsize=(16, 3.5))
ax.imshow(attn[:, :len(ctx)], cmap="viridis", aspect="auto")
ax.set_xticks(range(len(ctx)), ctx, rotation=90, fontsize=8)
ax.set_yticks(range(len(reply)), reply)
ax.set_title("attention: reply words (rows) vs. context tokens (columns)")
plt.tight_layout()
plt.show()

reply: you are olga, of course!


/var/folders/7p/wxlym5j148z5bvgtpk6tb9tm0000gn/T/ipykernel_88422/3623420417.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Save

In [10]:
MODEL_DIR.mkdir(exist_ok=True)
model.save(MODEL_DIR / "chatty.keras")
(MODEL_DIR / "vocab.json").write_text(json.dumps({"vocab": vocab, "max_context": MAX_CONTEXT,
                                                   "max_reply": MAX_REPLY, "memory": MEMORY}))
print("saved to", MODEL_DIR.resolve())

saved to /Users/vladislavshapovalov/projects/chatty-bot/model


## 8. Chat! 💬

`bot.say("...")` keeps the conversation history, so memory works across calls.
`bot.memory()` shows what the bot can currently "see".

Chatty only knows words from its training data (about 300). Unknown words
become `<unk>`, so tell it a name/food/pet/city from its lists. Run
`print(NAMES)` to see the names it knows.

In [11]:
class Chatty:
    def __init__(self, temperature=0.0):
        self.history, self.temperature = [], temperature

    def say(self, message: str) -> str:
        unknown = [t for t in tokenize(message) if t not in word2id]
        reply, _ = generate(context_tokens(self.history, message), self.temperature)
        reply = detokenize(reply)
        self.history.append({"user": message, "bot": reply})
        note = f"   (never seen: {', '.join(unknown)})" if unknown else ""
        print(f"🧑 {message}{note}\n🤖 {reply}\n")
        return reply

    def memory(self):
        for ex in self.history[-MEMORY:]:
            print(f"  🧑 {ex['user']}\n  🤖 {ex['bot']}")

    def reset(self):
        self.history = []


bot = Chatty()
for msg in ["hi", "my name is vlad", "i love pizza", "what's my name ?", "tell me a joke",
            "another joke", "what do i like to eat ?", "are you a robot ?", "who am i ?", "bye"]:
    bot.say(msg)

🧑 hi
🤖 hi there, human!



🧑 my name is vlad
🤖 nice to meet you, vlad!



🧑 i love pizza
🤖 pizza? yummy!



🧑 what's my name ?
🤖 you are vlad, of course!



🧑 tell me a joke
🤖 why did the robot go on holiday? to recharge its batteries!



🧑 another joke
🤖 what do you call a sleeping bull? a bulldozer!



🧑 what do i like to eat ?
🤖 hmm... i don't remember. tell me again?



🧑 are you a robot ?
🤖 beep boop. i mean, no. i mean, yes.



🧑 who am i ?
🤖 hmm... i don't remember. tell me again?



🧑 bye
🤖 bye bye! don't forget me



Interactive loop. Type `quit` to stop.

In [12]:
def chat_loop():
    bot = Chatty()
    while True:
        try:
            message = input("you: ").strip()
        except Exception:   # e.g. running headless without stdin
            print("(interactive input not available here; use bot.say(...) instead)")
            return
        if message.lower() in {"quit", "exit", "q"}:
            print("🤖 bye bye ! don't forget me")
            return
        if message:
            bot.say(message)


chat_loop()

(interactive input not available here; use bot.say(...) instead)
